<a href="https://colab.research.google.com/github/ubsuny/PHY386/blob/Homework2026/2026/FinalProject/Final_02_RingOfFire.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Final Project — Topic 2: Ring of Fire from Earthquake Data (42 pts)

## Learning Outcomes
- Query a public geophysical database (USGS ComCat) with the `libcomcat` Python client — same programmatic spirit as `astroquery` in HW6.
- Engineer features (lat, lon, depth, magnitude) from a real-world catalog.
- Label events by geographic membership in the Pacific Ring of Fire.
- Train k-Nearest Neighbors and Random Forest classifiers; compare them quantitatively.
- Connect the feature importance back to the physics of subduction zones and mid-ocean ridges.
- Visualize decision regions on a world map with `cartopy`.

## GitHub Workflow
Fork → `yourname-final` → pull request into `Homework2026`. Reviewer `@laserlab`, milestone `Final-2026`. See [`pull_request_template_Final.md`](pull_request_template_Final.md).

## Background: The Ring of Fire

The **Pacific Ring of Fire** is the horseshoe-shaped belt of tectonic activity around the Pacific Ocean — the convergent-plate boundaries where the Pacific, Nazca, Cocos, and Juan de Fuca plates subduct beneath the surrounding continental and oceanic plates. About **90% of the world's earthquakes** and most of its volcanoes occur in this belt.

Two physical predictions we can test:
1. **Location dominates.** Earthquakes cluster along plate boundaries. A classifier trained only on (latitude, longitude) should already get most of the signal.
2. **Depth distinguishes.** Subduction zones produce earthquakes at all depths, from shallow (<70 km) at the trench to deep-focus (>300 km) along the Wadati–Benioff zone. Mid-ocean ridges, in contrast, only produce shallow quakes. So including **depth** as a feature should separate subduction from other seismic activity.

You will check both.

The data come from the **USGS Comprehensive Catalog (ComCat)** via the official `libcomcat` Python package. Think of it as `astroquery` for earthquakes.

In [ ]:
# Required packages (uncomment for Colab or fresh environment):
# %pip install libcomcat-python cartopy scikit-learn --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

plt.rcParams.update({
    'font.size': 11, 'axes.labelsize': 11, 'axes.titlesize': 11,
    'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 11,
})

## Worked Example: Query USGS ComCat
`libcomcat.search.search` returns a list of event summaries. Each event has `latitude`, `longitude`, `depth` (km), and `magnitude`. We pull five years of M5+ events worldwide — a few thousand rows, comfortable for scikit-learn on a laptop.

In [ ]:
from libcomcat.search import search

events = search(
    starttime=datetime(2020, 1, 1),
    endtime=datetime(2024, 12, 31),
    minmagnitude=5.0,
)
df = pd.DataFrame(
    [{"lat": e.latitude, "lon": e.longitude, "depth": e.depth, "mag": e.magnitude}
     for e in events]
)
print(f"{len(df)} events")
df.head()

## Part 1 — Labeling Ring-of-Fire Membership (8 pts)

### Task 1.1 — Define a Ring-of-Fire polygon (4 pts)
Build a simple binary label:

$$\text{ring} = \begin{cases} 1 & \text{if event is in the Pacific Rim region} \\ 0 & \text{otherwise} \end{cases}$$

A coarse polygon is fine. Example rule (adjust if you want a tighter match):
- Western arm: longitude in [120°E, 180°E] and latitude in [−55°, 60°]
- Eastern arm: longitude in [−180°, −70°] and latitude in [−55°, 60°]

Remember that longitudes in the data are in [−180°, +180°].

### Task 1.2 — Visualize the labeling on a world map (4 pts)
Use `cartopy.crs.PlateCarree()` to plot all events color-coded by label. Sanity-check: does your polygon actually enclose the expected countries (Chile, Peru, Mexico, Japan, Indonesia, New Zealand, …)?

In [ ]:
# Part 1: your code here
# import cartopy.crs as ccrs
# import cartopy.feature as cfeature
# fig, ax = plt.subplots(figsize=(10, 5), subplot_kw={'projection': ccrs.PlateCarree()})
# ax.add_feature(cfeature.LAND, facecolor='lightgray')
# ax.scatter(df['lon'], df['lat'], c=df['ring'], s=4, cmap='RdBu_r',
#            transform=ccrs.PlateCarree())
# ...

## Part 2 — Baseline: Geography Only (8 pts)

### Task 2.1 — k-Nearest Neighbors on (lat, lon) (4 pts)
Features: `[lat, lon]`. Label: `ring`. Standardize with `StandardScaler`, split 80/20, train a k-Nearest Neighbors classifier with $k$ chosen by simple grid search over {3, 5, 9, 15}. Report accuracy and the confusion matrix on the test set.

### Task 2.2 — Interpretation (4 pts)
Why does k-Nearest Neighbors do well here even without depth? What would you expect if you moved to a dataset with different spatial coverage (e.g. M4+ events, which densely sample continental-interior seismicity)?

In [ ]:
# Part 2: your code here

## Part 3 — Adding Depth and Magnitude (12 pts)

### Task 3.1 — Random Forest on (lat, lon, depth, mag) (5 pts)
Re-train using all four features. Random Forest tolerates mixed scales, so you do not strictly need to standardize, but do it anyway for apples-to-apples comparison with Task 2. Report accuracy + confusion matrix.

### Task 3.2 — Feature importances (4 pts)
Plot the Random Forest's `feature_importances_` as a bar chart. Does depth actually help? By how much?

### Task 3.3 — Depth distribution by class (3 pts)
Side-by-side histograms of event depth for Ring-of-Fire vs. non-Ring events. Overlay the standard depth ranges:
- Shallow: 0–70 km
- Intermediate: 70–300 km
- Deep: >300 km

Comment on what you see.

In [ ]:
# Part 3: your code here

## Part 4 — Decision Regions on the Map (8 pts)

### Task 4.1 — Dense (lat, lon) grid (3 pts)
Build a grid of candidate points covering the globe at ~2° resolution.

### Task 4.2 — Predict and overlay (5 pts)
For each grid point, predict Ring-of-Fire membership with your k-Nearest Neighbors model (and separately with Random Forest — fix depth and magnitude at their median values). Overlay the predicted region on a `cartopy` world map using semi-transparent filled contours. Add coastlines. Compare the shape to the real Ring-of-Fire.

In [ ]:
# Part 4: your code here

## Part 5 — Physics Discussion (6 pts)

Write 5–8 sentences addressing:
1. Which feature was most informative — latitude, longitude, depth, or magnitude? Does that match the physics of subduction vs. mid-ocean ridge seismicity?
2. Where did your classifier get it wrong? (e.g. the Mediterranean is seismically active but not in the Ring of Fire.)
3. If you wanted to recover the Ring-of-Fire *without* the polygon-based labels — i.e. unsupervised — what would you try? (Hint: think DBSCAN or KMeans in (lat, lon, depth) space.)

### Stretch (optional, bonus 4 pts)
Download the Bird (2003) tectonic-plate-boundary shapefile and overlay it on your decision-region plot. Comment on which boundary types (convergent, divergent, transform) your classifier captures well vs. poorly.

## Submission
**Good commit** ✅: `Add Random Forest model with depth and magnitude features`

**Bad commit** ❌: `fix`, `stuff`, `done`

### Checklist
- [ ] Notebook in `2026/FinalProject/<yourname>/`
- [ ] Runs top-to-bottom on a fresh kernel
- [ ] Type annotations and NumPy docstrings on your functions
- [ ] All plots labeled with units
- [ ] Tasks sum to ~42 points
- [ ] PR against `Homework2026`, reviewer `@laserlab`, milestone `Final-2026`
- [ ] AI usage disclosed in the PR description

## Resources
- [`libcomcat` documentation](https://code.usgs.gov/ghsc/esi/libcomcat-python)
- [USGS ComCat search parameters](https://earthquake.usgs.gov/fdsnws/event/1/)
- [Cartopy tutorial](https://scitools.org.uk/cartopy/docs/latest/tutorials/understanding_transform.html)
- Bird, P. (2003). *An updated digital model of plate boundaries.* Geochem. Geophys. Geosyst., 4(3).
- PHY386 plotting-style reference: [`2026/handson/ScientificPlottingMatplotlib.ipynb`](../handson/ScientificPlottingMatplotlib.ipynb)